[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_cytotoxicity_filter.ipynb)

# Filter the shape hits by predicted toxicity

**Blue group · Cryptosporidiosis**

The shape-similarity notebook left roughly two thousand molecules, and we need fewer. Here we
narrow them to a thousand by asking a different question from the other filters: not whether a
molecule is likely to hit CpABC1, but whether it is likely to harm human cells. A compound
that kills the patient's cells as well as the parasite is no use, however well it binds.

## What you will do

- Load the shape hits together with their predicted toxicity to three human cell types.
- Look at how toxic the molecules are, and at whether the three cell types agree.
- Keep the thousand that look least toxic to liver cells.
- Check what that filter changed, and look at the molecules it threw away.
- Save the shortlist for the next step.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The molecules and their predicted toxicity

Two files come together here. The predictions are stored in the repository, so there is
nothing to download. The list of molecules from the shape-similarity notebook is not, because
you produced it, so in Colab the cell below asks you to upload it. We need it for the MolPort
catalogue numbers, which are how you would actually buy a compound.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

DOWNLOADS = Path("data/downloads")
DOWNLOADS.mkdir(parents=True, exist_ok=True)
HITS = DOWNLOADS / "sand_filtered_hits.csv"

if "google.colab" in sys.modules and not HITS.exists():
    from google.colab import files
    print("Upload sand_filtered_hits.csv, saved by the shape-similarity notebook")
    for name in files.upload():
        Path(name).rename(HITS)

toxicity = pd.read_csv("data/eos42ez_sand_hits.csv")
hits = pd.read_csv(HITS)
print(f"{len(toxicity)} predictions, {len(hits)} molecules")

The predictions carry the molecule as a column called `input`, so we join the two tables
on the SMILES. The check afterwards matters: if the join loses rows, the two files describe
different sets of molecules and everything below would be quietly wrong.

In [ ]:
CELL_LINES = ["cytotoxicity_hepg2", "cytotoxicity_hskmc", "cytotoxicity_imr90"]

data = hits.merge(toxicity[["input"] + CELL_LINES],
                  left_on="smiles", right_on="input", how="inner").drop(columns="input")
assert len(data) == len(hits), f"join kept {len(data)} of {len(hits)} molecules"

print(f"{len(data)} molecules with predictions")
data.head()

## 2. What the model predicts

The predictions come from [eos42ez](https://github.com/ersilia-os/eos42ez) in the Ersilia
Model Hub, a model built from the work published in
[Nature in 2023](https://doi.org/10.1038/s41586-023-06887-8). It was trained on 39,312
compounds that were put on human cells at a concentration of 10 micromolar, where a compound
counted as toxic if fewer than 90 in every 100 cells survived.

It reports one number per cell type, between 0 and 1, and **higher means more likely to be
toxic**. That is the opposite direction to most of the scores in these notebooks, so the sort
below runs the other way.

- `cytotoxicity_hepg2` — liver cells
- `cytotoxicity_hskmc` — skeletal muscle cells
- `cytotoxicity_imr90` — lung cells

> **Note:** This is a prediction about harm, not about activity. It says nothing at all about
> whether a molecule reaches CpABC1. A molecule that survives this filter is one that looks
> less likely to be toxic, which is a reason to keep it, not evidence that it works.

## 3. How toxic do these molecules look?

Start with the liver cells, which is the number we will filter on. The histogram shows the
whole set at once.

In [ ]:
import stylia

figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
ax.hist(data["cytotoxicity_hepg2"], bins=60, color=stylia.NamedColors().cobalt)
stylia.label(ax, xlabel="Predicted toxicity to liver cells (0 to 1)",
             ylabel="Number of molecules")
figure.tight_layout()

Most of the set sits low, which is reassuring, and a tail reaches higher. The table below
gives the same picture as numbers, for all three cell types.

In [ ]:
data[CELL_LINES].describe().round(3).T

## 4. Do the three cell types agree?

We are about to throw away 887 molecules on the strength of one number, so it is worth asking
whether the other two would have agreed with it.

In [ ]:
data[CELL_LINES].corr().round(3)

Liver and muscle cells agree almost completely, so using one of them is much like using
both. Lung cells are a weaker match, which means they carry information the other two do not.
The plot below shows both comparisons side by side.

In [ ]:
nc = stylia.NamedColors()

figure, axes = stylia.create_figure(1, 2, width=1.0, height=0.5)
for other, colour in [("cytotoxicity_hskmc", "cobalt"), ("cytotoxicity_imr90", "tangerine")]:
    ax = axes.next()
    ax.scatter(data["cytotoxicity_hepg2"], data[other], s=6, alpha=0.3, color=nc.get(colour))
    r = data["cytotoxicity_hepg2"].corr(data[other])
    stylia.label(ax, xlabel="liver cells", ylabel=other.replace("cytotoxicity_", ""),
                 title=f"correlation {r:.2f}")
figure.tight_layout()

> **Note:** Because liver and muscle agree so closely, filtering on liver cells alone
> effectively covers both. Lung cells are the disagreement, so a molecule that looks safe here
> could still be predicted toxic to them. We accept that to keep the filter simple, but it is
> a real limitation and worth remembering when these thousand molecules are looked at again.

> **Exercise:** Rerun section 5 sorting on `cytotoxicity_imr90` instead. How many of the
> thousand molecules are the same ones? If the answer is most of them, the choice of cell type
> hardly matters. If it is not, the choice matters and deserves more thought than we gave it.

## 5. Keep the thousand least toxic

Now we sort so that the least toxic molecules come first, and keep the top thousand. Note
`ascending=True`: low scores are the good ones here.

In [ ]:
KEEP = 1000

ranked = data.sort_values("cytotoxicity_hepg2", ascending=True).reset_index(drop=True)
shortlist = ranked.head(KEEP).copy()

cutoff = shortlist["cytotoxicity_hepg2"].max()
print(f"kept {len(shortlist)} of {len(ranked)}, scores {cutoff:.3f} and below")
shortlist.head()

A quick check that the filter went the way we intended. The molecules we kept should have
a lower median score than the ones we dropped. If it comes out the other way round, the sort
is backwards and we have selected for toxicity.

In [ ]:
dropped = ranked.tail(len(ranked) - KEEP)

print(f"kept    median {shortlist['cytotoxicity_hepg2'].median():.3f}")
print(f"dropped median {dropped['cytotoxicity_hepg2'].median():.3f}")
assert shortlist["cytotoxicity_hepg2"].median() < dropped["cytotoxicity_hepg2"].median()

## 6. What did the filter change?

Any filter changes the kind of molecule you are left with. Comparing the molecules we kept
against the ones we dropped shows what this one selected for, besides toxicity itself.

Expect logP, which measures greasiness, to differ. Greasy molecules tend to damage cell
membranes, so a toxicity model learning to dislike them is the model working, not a fault.

In [ ]:
from scripts import chemspace

properties = chemspace.describe(ranked["smiles"])
kept = ranked.index < KEEP

pd.DataFrame({
    "kept": properties[kept].median(),
    "dropped": properties[~kept].median(),
}).round(1)

The same comparison as a picture, one panel per property, with silymarin marked as a line
for reference.

In [ ]:
from scripts import shape

seed = chemspace.describe(pd.read_csv("data/silymarin.csv")["smiles"])

figure, axes = stylia.create_figure(2, 3, width=1.0)
shape.plot_properties(axes, properties[kept], properties[~kept], seed)
figure.tight_layout()

## 7. The molecules we threw away

It is easy to look only at what a filter keeps. These are the twelve molecules predicted most
toxic to liver cells, the ones at the very bottom of the list. Look for anything they have in
common.

In [ ]:
worst = ranked.tail(12).iloc[::-1]

shape.draw_molecules(
    worst["smiles"].tolist(),
    [f"{row.molport_id}\ntoxicity {row.cytotoxicity_hepg2:.2f}"
     for row in worst.itertuples()])

## 8. Save the shortlist

Both files go into `outputs/`, and Colab downloads them to your computer. Keep the ranked file
as well as the shortlist: it holds the score of every molecule, which you will want if you
later decide a thousand was the wrong number.

In [ ]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
columns = ["molport_id", "smiles"] + CELL_LINES

ranked[columns].to_csv(OUT / "cytotoxicity_ranked.csv", index=False)
shortlist[columns].to_csv(OUT / "cytotoxicity_shortlist_1000.csv", index=False)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(OUT / "cytotoxicity_ranked.csv"))
    files.download(str(OUT / "cytotoxicity_shortlist_1000.csv"))
print(f"saved {len(ranked)} ranked and {len(shortlist)} shortlisted molecules")

## Summary

- Joined the shape-similarity hits to predicted toxicity for three human cell types, from the
  Ersilia model eos42ez.
- Found that liver and muscle predictions agree almost completely, while lung predictions
  disagree enough to carry their own information.
- Kept the thousand molecules predicted least toxic to liver cells, and checked that the
  filter selected mainly for less greasy molecules, which is what a toxicity model should do.
- Remember what this filter is: a way of avoiding molecules likely to harm human cells. It is
  not evidence that any of these thousand reach CpABC1.

**Next:** compare this shortlist with the one from the SPRINT notebook. Molecules that appear
in both were chosen for two unrelated reasons, which makes them the most interesting ones to
take into docking.